# Ingesta de países (incremental)

Lee `country.json` (JSON) del lote `incremental_append/<p_file_date>` en Bronze, aplica un esquema explícito, estandariza las columnas y agrega `file_date`. Reemplaza la tabla Delta `movie_silver.countries` en cada ejecución (`overwrite`), por lo que conserva solo el último lote procesado. Conserva los métodos alternativos del notebook de carga completa.

### Configuración

Carga las rutas y las funciones comunes, y lee los parámetros `p_environment` y `p_file_date` (fecha del lote).

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

### Lectura del archivo

Primero lee `country.json` con inferencia para inspeccionarlo y luego lo relee con un esquema explícito.

In [0]:
country_df = (
    spark.read
        .format('json')
        .option('inferSchema', True)
        .load(f'{bronze_folder_path}/incremental_append/{v_file_date}/country.json')
)

In [0]:
country_schema = "countryId string, countryIsoCode string, countryName string"
country_df = (
    spark.read
        .format('json')
        .option('header', True)
        .schema(country_schema)
        .load(f'{bronze_folder_path}/incremental_append/{v_file_date}/country.json')
)

### Eliminar columnas

#### Metodo 1

In [0]:
country_df_dropped = country_df.drop('countryIsoCode')

#### Metodo 2

In [0]:
country_df_dropped = country_df.drop(country_df['countryIsoCode'])

#### Metodo 3

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp

In [0]:
country_df_dropped = country_df.drop(col('countryIsoCode'))

### Renombrado y metadatos

Renombra `countryId` y `countryName` a snake_case y agrega `ingestion_date`, `env` y `file_date`.

In [0]:
country_df_final = (
    add_ingestion_date(country_df_dropped)
        .withColumnsRenamed({'countryId': 'country_id', 'countryName': 'country_name'})
        .withColumn('env', lit(v_env))
        .withColumn('file_date', lit(v_file_date))
)

### Escritura en Silver

Reemplaza la tabla Delta `movie_silver.countries` (modo `overwrite`).

In [0]:
(
    country_df_final.write
        .mode('overwrite')
        .saveAsTable('movie_silver.countries')
)

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")